# FIFA World Cup 2026 – Linear Regression

This notebook uses the match data for Objective 2.1 and the team-match data for Objective 2.2. I compare different train/test splits and use GroupKFold to check the results. The original CSVs are left as they are.

## 1. Load the data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

folder = Path('CSV_Files')
df21 = pd.read_csv(folder / 'FIFA2026_Objective2_1_Goal_Difference_104_Matches.csv')
df22 = pd.read_csv(folder / 'FIFA2026_Objective2_2_Goals_Scored_208_TeamMatches.csv')

print('2.1:', df21.shape, '2.2:', df22.shape)
display(df21.head(3))
display(df22.head(3))

## 2. Variables

Each model uses eight predictors. For 2.2, two of the original variables were replaced after we checked whether their values were known before the match. I calculate those replacements below instead of changing the CSV.

In [ ]:
x21 = [col for col in df21 if col.startswith('X')]
y21 = 'Y_GoalDifference'
y22 = 'Y_TeamGoals'
assert len(x21) == 8
print('2.1 predictors:', x21)


## 3. Clean and check the data

In [ ]:
def clean_data(df, columns, target):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    for col in columns + [target]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.sort_values(['Date', 'Match_ID']).reset_index(drop=True)

df21 = clean_data(df21, x21, y21)
old_x22 = [col for col in df22.columns if col.startswith('X')]
df22 = clean_data(df22, old_x22, y22)
print('Rows:', len(df21), 'for 2.1 and', len(df22), 'for 2.2')


In [ ]:
# Two replacement variables for Objective 2.2
# Use Age.csv and previous international matches. The input CSVs are not edited.
ages = pd.read_csv(folder / 'Age.csv', encoding='latin1')
history = pd.read_csv(folder / 'International_results.csv', encoding='latin1')
history['date'] = pd.to_datetime(history['date'], format='mixed', dayfirst=True, errors='coerce')

# A few team names are different between files.
team_names = {'Turkey': 'Turkiye', 'DR Congo': 'Democratic Republic of the Congo',
              'Bosnia and Herzegovina': 'Bosnia-Herzegovina'}
ages['Club'] = ages['Club'].str.strip()
wc_counts = dict(zip(ages['Club'], pd.to_numeric(ages['WC particip.'], errors='coerce')))

# Age.csv appears to count the 2026 tournament as one appearance for each team.
# Subtracting one gives the pre-2026 count. The source date still needs checking.
def previous_wc(team):
    name = team_names.get(team, team)
    count = wc_counts.get(name, np.nan)
    return count - 1 if pd.notna(count) else np.nan

df22['X3_Pre2026_WC_Participation_Difference'] = df22.apply(
    lambda row: previous_wc(row['Team']) - previous_wc(row['Opponent']), axis=1)

# Loss rate of the opposing team in its seven matches before each fixture.
# First take matches from the historical file before the World Cup started.
start = pd.Timestamp('2026-06-11')
history = history[history['date'].notna() & (history['date'] < start)].copy()

aliases = {'Turkiye':'Turkey', 'Türkiye':'Turkey', 'Bosnia-Herzegovina':'Bosnia and Herzegovina',
           'Democratic Republic of the Congo':'DR Congo', 'Czech Republic':'Czechia',
           'Curacao':'Curaçao', 'United States of America':'United States'}
def same_name(team):
    return aliases.get(str(team).strip(), str(team).strip())

# Each entry records one team's result as 1 (loss) or 0 (not a loss).
results = {}
for r in history.itertuples(index=False):
    home, away = same_name(r.home_team), same_name(r.away_team)
    results.setdefault(home, []).append((r.date, int(r.home_score < r.away_score)))
    results.setdefault(away, []).append((r.date, int(r.away_score < r.home_score)))

# Read the earlier 2026 games in order, so later matches can use their results.
# Both team rows belonging to one match are updated only after its features are calculated.
df22['X3_Pre2026_WC_Participation_Difference'] = pd.to_numeric(
    df22['X3_Pre2026_WC_Participation_Difference'], errors='coerce')
loss_rates = {}
for match_id, match in df22.groupby('Match_ID', sort=False):
    match_date = match['Date'].iloc[0]
    for row in match.itertuples():
        opponent = same_name(row.Opponent)
        previous = [lost for date, lost in results.get(opponent, []) if date < match_date]
        loss_rates[row.Index] = np.mean(previous[-7:]) if previous else np.nan
    for row in match.itertuples():
        team = same_name(row.Team)
        opponent_goals = match.loc[match['Team'] == row.Opponent, y22].iloc[0]
        results.setdefault(team, []).append((match_date, int(row.Y_TeamGoals < opponent_goals)))

df22['X1_Opponent_Loss_Rate_Last7'] = pd.Series(loss_rates)
# Keep the original order of Manish's four variables, then variables 5-8.
x22 = ['X2_PreMatch_Computed_Elo_Diff',
       'X3_Pre2026_WC_Participation_Difference',
       'X1_Opponent_Loss_Rate_Last7',
       'X4_Past7_UnbeatenRate_Team',
       'X5_Prior_WC_Team_AvgGoals', 'X6_Prior_WC_Opp_AvgConceded',
       'X7_Avg_SquadAge_Team', 'X8_DaysSincePrevInternational_Team']
assert len(x22) == 8 and df22['X3_Pre2026_WC_Participation_Difference'].notna().all()
print('2.2 predictors:', x22)
print('Missing opponent loss rates:', df22['X1_Opponent_Loss_Rate_Last7'].isna().sum())
print('All input CSV files remain unchanged.')


The World Cup participation numbers come from `Age.csv`. The file does not show the date of the snapshot, so the pre-2026 counts still need checking against a dated source. The opponent loss rate uses the opponent’s last seven international results before each match; one missing value is filled during model training.

## 4. Descriptive statistics and scatterplots

In [ ]:
display(df21[x21 + [y21]].describe().round(2))
display(df22[x22 + [y22]].describe().round(2))

def scatterplots(df, features, target, title):
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    for ax, col in zip(axes.flat, features):
        ax.scatter(df[col], df[target], alpha=0.55, s=16)
        ax.set_title(col.replace('_', ' '), fontsize=9)
        ax.set_xlabel('Predictor', fontsize=8)
        ax.set_ylabel(target, fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

scatterplots(df21, x21, y21, 'Objective 2.1: Predictor scatterplots')
scatterplots(df22, x22, y22, 'Objective 2.2: Predictor scatterplots')

## 5. Correlation check

This helps spot predictors that are closely related. Correlation does not prove that a variable is useful for prediction.

In [ ]:
def correlations(df, features, target, title):
    corr = df[features + [target]].corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(len(corr.columns)), [str(i+1) for i in range(len(corr))])
    ax.set_yticks(range(len(corr.columns)), [str(i+1) for i in range(len(corr))])
    ax.set_title(title + ' (1–8 = X variables; 9 = target)')
    fig.colorbar(im, ax=ax)
    plt.show()
    display(corr[target].drop(target).to_frame('Correlation with target').round(3))

correlations(df21, x21, y21, 'Objective 2.1')
correlations(df22, x22, y22, 'Objective 2.2')

## 6. Build and test the models

I tried 60/40, 70/30 and 80/20 train/test splits to see how the results changed. I kept the two rows from each match together in 2.2. I also tested on the later 20% of matches separately. Missing predictor values were filled using the training data medians only.


In [ ]:
def run_model(data, features, target, test_size=0.2, chronological=False):
    matches = data[['Match_ID', 'Date']].drop_duplicates('Match_ID').sort_values(['Date','Match_ID'])
    if chronological:
        cut = int(len(matches) * (1 - test_size))
        train_ids = set(matches.iloc[:cut]['Match_ID'])
        test_ids = set(matches.iloc[cut:]['Match_ID'])
    else:
        train_ids, test_ids = train_test_split(matches['Match_ID'].to_numpy(), test_size=test_size, random_state=42)
        train_ids, test_ids = set(train_ids), set(test_ids)

    train = data[data['Match_ID'].isin(train_ids)].copy()
    test = data[data['Match_ID'].isin(test_ids)].copy()
    # A match must never appear in both sets.
    assert not (train_ids & test_ids)
    medians = train[features].median().fillna(0)
    X_train = train[features].fillna(medians)
    X_test = test[features].fillna(medians)
    y_train, y_test = train[target], test[target]
    # Targets cannot be filled because they are the actual results.
    valid_train = y_train.notna()
    valid_test = y_test.notna()
    model = LinearRegression()
    model.fit(X_train.loc[valid_train], y_train.loc[valid_train])
    prediction = model.predict(X_test.loc[valid_test])
    actual = y_test.loc[valid_test]
    metrics = {
        'Train rows': int(valid_train.sum()), 'Test rows': int(valid_test.sum()),
        'MAE': mean_absolute_error(actual, prediction),
        'RMSE': np.sqrt(mean_squared_error(actual, prediction)),
        'R2': r2_score(actual, prediction)
    }
    return metrics, model, actual.to_numpy(), prediction, train, test, medians

splits = [('60/40',0.4,False), ('70/30',0.3,False), ('80/20',0.2,False),
          ('Chronological 80/20',0.2,True)]
results = []
final_models = {}
for label, data, cols, target in [('2.1',df21,x21,y21), ('2.2',df22,x22,y22)]:
    for split_name, size, chrono in splits:
        metrics, model, actual, pred, train, test, medians = run_model(data,cols,target,size,chrono)
        results.append({'Objective':label,'Split':split_name,**metrics})
        if chrono:
            final_models[label] = (model,actual,pred,train,test,medians,cols,target)

scores = pd.DataFrame(results)
display(scores.round(3))

## 7. GroupKFold cross-validation

I added 5-fold GroupKFold to check whether the models give similar results across different sets of matches. It groups by match ID, which is important for 2.2 because each match has two team rows.

I used only the training matches from the chronological 80/20 split for cross-validation. The test matches were kept aside. Missing values were filled separately within each training fold.


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cv_results = []
for label, data, cols, target in [('2.1', df21, x21, y21), ('2.2', df22, x22, y22)]:
    # Use the earlier matches from the final chronological split.
    train = final_models[label][3]
    train = train.dropna(subset=[target]).copy()
    X = train[cols]
    y = train[target]
    groups = train['Match_ID']
    folds = GroupKFold(n_splits=5)

    for fold_no, (train_idx, val_idx) in enumerate(folds.split(X, y, groups), start=1):
        model = make_pipeline(SimpleImputer(strategy='median'), LinearRegression())
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predicted = model.predict(X.iloc[val_idx])
        actual = y.iloc[val_idx]
        cv_results.append({
            'Objective': label, 'Fold': fold_no,
            'Training matches': groups.iloc[train_idx].nunique(),
            'Validation matches': groups.iloc[val_idx].nunique(),
            'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)
        })

cv_scores = pd.DataFrame(cv_results)
print('Five-fold results:')
display(cv_scores.round(3))
print('Average results (lower MAE/RMSE is better):')
cv_summary = cv_scores.groupby('Objective')[['MAE', 'RMSE', 'R2']].agg(['mean', 'std'])
display(cv_summary.round(3))

# Save this table separately; the original CSV files are unchanged.
output_folder = Path('Outputs')
output_folder.mkdir(exist_ok=True)
cv_scores.to_csv(output_folder / 'GroupKFold_5Fold_Results.csv', index=False)


## 8. Final test plots and residuals


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for i, label in enumerate(['2.1','2.2']):
    model, actual, pred, train, test, medians, cols, target = final_models[label]
    residuals = actual - pred
    ax = axes[i,0]
    ax.scatter(actual,pred,alpha=0.7)
    bounds = [min(actual.min(),pred.min()), max(actual.max(),pred.max())]
    ax.plot(bounds,bounds,'--',color='grey')
    ax.set(xlabel='Actual',ylabel='Predicted',title=f'{label}: Actual vs predicted')
    ax = axes[i,1]
    ax.scatter(pred,residuals,alpha=0.7)
    ax.axhline(0,linestyle='--',color='grey')
    ax.set(xlabel='Predicted',ylabel='Residual (actual − predicted)',title=f'{label}: Residual plot')
fig.tight_layout()
plt.show()

## 9. Final checks

I checked the predictions, residuals and model coefficients. The pre-2026 participation counts still need a dated source to confirm they were known before the tournament.

In [ ]:
for label in ['2.1','2.2']:
    model, actual, pred, train, test, medians, cols, target = final_models[label]
    residuals = actual - pred
    print(f'\nObjective {label}:')
    print('Test residual mean:', round(float(residuals.mean()), 3))
    print('Test residual standard deviation:', round(float(residuals.std()), 3))
    print('Maximum absolute predictor correlation:', round(float(train[cols].corr().abs().where(~np.eye(len(cols),dtype=bool)).max().max()),3))
    print('Regression coefficients:')
    display(pd.DataFrame({'Variable':cols, 'Coefficient':model.coef_}).round(3))
print('\nThe date of the World Cup participation counts still needs checking; market value is also used in 2.1.')

In [ ]:
# Save updated model scores (original CSV data remains unchanged).
output_folder.mkdir(exist_ok=True)
scores.to_csv(output_folder / 'Train_Test_Split_Results.csv', index=False)
